<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 25 · ML for Yield Curves and Rates

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch25_ml_yield_curves_rates.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

Rates are persistent and noisy, so the no-change forecast is the benchmark to beat. You will follow the chapter script through the full construction: level, slope, and curvature features from four weekly yield nodes quoted in percent, a target defined as next week's 10-year yield change in percentage points, purging of training labels that reach the holdout, and a ridge forecast compared with a zero-change random walk. You will also summarize calm, transition, and stress regimes from the frozen regime panel and practice reporting forecast errors and regime diagnostics without overstating predictive power.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script builds curve-factor features from four weekly yield nodes, defines the target as next week's 10-year yield change, and forecasts the final four holdout dates with both a ridge regression and a zero-change random walk. The ridge forecasts run between 0.024 and 0.078 percentage points against realized targets from -0.10 to 0.33, and the second table reports mean level and slope for calm, transition, and stress regimes.

In [ ]:
SCRIPT = ROOT / "code" / "ch25_ml_yield_curves_rates.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Read both printed tables with their units.

- Yields are quoted in percent, so targets and forecasts are next-week 10-year yield changes in percentage points.
- The random-walk forecast is exactly 0.0 on every holdout date; score the ridge forecast against it before crediting the model.
- The four holdout targets are 0.22, -0.10, 0.33, and 0.23; the ridge forecasts are 0.035, 0.024, 0.031, and 0.0781.
- The regime summary orders mean level and slope monotonically from calm (3.51, 0.835) to stress (4.1981, 1.345); regimes describe environments, not forecasts.
- The weekly panel is synthetic teaching data; the official US Treasury curve ML panel is a separate frozen file for the same workflow.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch25_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Mini-lab: Lagged curve features that avoid look-ahead

A daily curve-level series follows a seeded random walk in percent. The target is the next day's change, which is unknown at prediction time. Two candidate features are built from daily changes: a leaky one shifted so that it overlaps the target window, and a properly lagged one that uses only the change already observed at the prediction date. Their correlations with the target expose the difference: the leaky feature is mechanically tied to the target, while the lagged one is not. Correlations, not forecasts, are the output here.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(3)  # fixed seed for the level path
dates = pd.date_range("2026-01-02", periods=24, freq="B")
level = pd.Series(3.0 + np.cumsum(rng.normal(0.0, 0.06, 24)),
                  index=dates)  # curve level in percent
target = level.shift(-1) - level  # next-day change
leaky = level.diff().shift(-1)  # overlaps the target window
lagged = level.diff().shift(1)  # known at prediction time
corr = pd.DataFrame({"target": target, "leaky": leaky,
                     "lagged": lagged}).corr()["target"]
print(f"leaky correlation: {corr['leaky']:0.4f}")
print(f"lagged correlation: {corr['lagged']:0.4f}")

### What to observe

The leaky feature correlates with the target at exactly 1.0000 because it is built from the same future change the target measures; the properly lagged feature manages only 0.0628 on this seeded random walk. A feature that looks this good against a random walk is not skill, it is look-ahead.

## Exercises

Work these with the script output and the frozen panels; keep units and split hygiene explicit.

1. **Rebuild the features.** Recompute level, slope, and curvature from the four yield nodes using the script's definitions. *Hint:* Level is the cross-node mean, slope is the 10-year minus the 1-year yield, curvature is twice the 5-year minus the 2-year minus the 10-year.
2. **Score against the random walk.** Compare the mean absolute errors of the ridge forecasts and the zero-change forecast across the four holdout dates. *Hint:* Both forecast columns are printed next to the realized target.
3. **Explain the purge.** Identify which training row the script drops before fitting, and why. *Hint:* Its target end date falls on the first holdout date.
4. **Read the regimes.** Order calm, transition, and stress by mean level and slope, and describe the risk discussion each regime supports. *Hint:* The regime summary table already holds the means.
5. **Hunt look-ahead.** Suppose a feature uses the same week's 10-year yield change as the target; explain why a chronological split alone would not contain the damage. *Hint:* The feature and the label share one outcome window.
6. **Check the scaling.** Confirm that scaling parameters are computed from training rows only. *Hint:* The script fits mean and standard deviation on the training matrix before the holdout is standardized.

For more detail, see Chapter 25 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">